In [1]:
import os
import sys
import scipy
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
import scipy.io as sio
import scanpy.external as sce
import matplotlib.pyplot as plt
import re
import anndata as ad
import statistics
import torch
import scvi
import tempfile
import sklearn
import mudata as md
import muon as mu
md.set_options(pull_on_update=False)
import muon
from datetime import datetime
from scib_metrics.benchmark import Benchmarker
from sklearn_ann.kneighbors.annoy import AnnoyTransformer
from multiprocessing import Pool
import matplotlib.colors as mcolors
import uuid
print("Last run with scvi-tools version:", scvi.__version__)
scvi.settings.num_threads = 16
scvi.settings.seed = 0
sc._settings.n_jobs=16
sc.settings.verbosity = 4
sc.settings.set_figure_params(dpi=100, fontsize=10, dpi_save=400,
    facecolor = 'white', figsize=(8,8), format='pdf')
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 100)

import matplotlib.pyplot as plt
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams['lines.linewidth'] = 0.5

/home/liyanguo/anaconda3/envs/R45/lib/python3.13/site-packages/scanpy/_utils/__init__.py:33: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  from anndata import __version__ as anndata_version
/home/liyanguo/anaconda3/envs/R45/lib/python3.13/site-packages/scanpy/__init__.py:24: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  if Version(anndata.__version__) >= Version("0.11.0rc2"):
/home/liyanguo/anaconda3/envs/R45/lib/python3.13/site-packages/scanpy/readwrite.py:16: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  if Version(anndata.__version__) >= Version("0.11.0rc2"):
Seed set to 0


Last run with scvi-tools version: 1.3.3


In [2]:
dataset='Full_dataset'

In [3]:
obj_path = f'/home/liyanguo/MyImmuCell/06_Finnal_raw_count/{dataset}/'
sc.settings.figdir = obj_path

In [ ]:
adata = sc.read_h5ad(f"{obj_path}{dataset}_scRNA_count_HVG.h5ad")
adt = sc.read_h5ad(f"{obj_path}{dataset}_scADT_count.h5ad")

In [5]:
adata = sc.read_h5ad(f"{obj_path}{dataset}_scRNA_count.h5ad",backed=True)

In [ ]:
indices = pd.read_csv("/home/liyanguo/MyImmuCell/05_MyImmuCell_subpopulation/Final_cell_label.csv",index_col=0)

In [ ]:
indices = indices[['Classification_L4','Classification_L3','Classification_L2','Classification_L1','leiden_cluster']]

In [ ]:
if (adata.obs.index == indices.index).all():
    adata.obs = adata.obs.join(indices,how='left')
    adt.obs = adt.obs.join(indices, how='left')
else:
    raise ValueError('Indices error.')

In [ ]:
adata.strings_to_categoricals()

# 1. Load latent representation and umap

In [9]:
adata.obsm["X_TOTALVI"]=np.load(f"{obj_path}{dataset}_X_TOTALVI.npy")

In [10]:
print(f"Do sc.pp.neighbors with AnnoyTransformer,{datetime.now()}")
sc.pp.neighbors(adata, transformer=AnnoyTransformer(20), use_rep='X_TOTALVI')

Do sc.pp.neighbors with AnnoyTransformer,2025-11-07 09:17:18.724927
computing neighbors
    computing neighbors


/home/liyanguo/anaconda3/envs/R45/lib/python3.13/site-packages/scanpy/neighbors/__init__.py:430: FutureWarning: Use obsm (e.g. `k in adata.obsm` or `adata.obsm.keys() | {'u'}`) instead of AnnData.obsm_keys, AnnData.obsm_keys is deprecated and will be removed in the future.
  if "X_diffmap" in adata.obsm_keys():


    computed neighbors (1:00:47)
    computed connectivities (0:06:55)
    finished: added to `.uns['neighbors']`
    `.obsp['distances']`, distances for each pair of neighbors
    `.obsp['connectivities']`, weighted adjacency matrix (1:07:42)


In [11]:
print(f"Run UMAP,{datetime.now()}")
sc.tl.umap(adata, min_dist=0.5)

Run UMAP,2025-11-07 10:25:01.764182
computing UMAP


  0%|          | 0/200 [00:00<?, ?it/s]

	completed  0  /  200 epochs
	completed  20  /  200 epochs
	completed  40  /  200 epochs
	completed  60  /  200 epochs
	completed  80  /  200 epochs
	completed  100  /  200 epochs
	completed  120  /  200 epochs
	completed  140  /  200 epochs
	completed  160  /  200 epochs


IOStream.flush timed out
IOStream.flush timed out
IOStream.flush timed out
IOStream.flush timed out
IOStream.flush timed out
IOStream.flush timed out
IOStream.flush timed out
IOStream.flush timed out
IOStream.flush timed out
IOStream.flush timed out
IOStream.flush timed out
IOStream.flush timed out
IOStream.flush timed out
IOStream.flush timed out
IOStream.flush timed out
IOStream.flush timed out
IOStream.flush timed out
IOStream.flush timed out
IOStream.flush timed out
IOStream.flush timed out
IOStream.flush timed out
IOStream.flush timed out
IOStream.flush timed out
IOStream.flush timed out
IOStream.flush timed out
IOStream.flush timed out
IOStream.flush timed out
IOStream.flush timed out
IOStream.flush timed out
IOStream.flush timed out


	completed  180  /  200 epochs
    finished: added
    'X_umap', UMAP coordinates (adata.obsm)
    'umap', UMAP parameters (adata.uns) (19:03:15)


In [12]:
#Not save
#adata.write(f"{obj_path}{dataset}_scRNA_count_HVG.h5ad",compression="gzip")

In [13]:
indices['UMAP_1'] = adata.obsm['X_umap'][:, 0]  #
indices['UMAP_2'] = adata.obsm['X_umap'][:, 1]  #

In [14]:
indices.head()

,Classification_L4,Classification_L3,Classification_L2,Classification_L1,leiden_cluster,UMAP_1,UMAP_2
D0110_E_Rep1_CCGAAGTA_AGAGTCAA_ATCATTCC,Basophils,Basophils,Granulocytes,Myeloid cells,Basophil c0,6.692915,10.123938
D0110_E_Rep1_CTGAGCCA_GAACAGGC_ACGCTCGA,Basophils,Basophils,Granulocytes,Myeloid cells,Basophil c0,6.640472,10.080825
D0110_E_Rep1_AACGCTTA_CAAGACTA_GGTGCGAA,Basophils,Basophils,Granulocytes,Myeloid cells,Basophil c0,6.492055,10.113962
D0110_E_Rep1_CGCTGATC_GCCACATA_CAGCGTTA,Basophils,Basophils,Granulocytes,Myeloid cells,Basophil c0,6.413828,10.019212
D0110_E_Rep1_CATCAAGT_TATCAGCA_AGTACAAG,Basophils,Basophils,Granulocytes,Myeloid cells,Basophil c0,7.053688,9.903442


In [15]:
indices.to_csv(f"{obj_path}/{dataset}_indices_labels_umap.csv")

# 2. Plot leiden umap

In [4]:
adata = sc.read_h5ad(f"{obj_path}{dataset}_scRNA_count_HVG.h5ad",backed=True)

In [19]:
indices = pd.read_csv(f"{obj_path}/{dataset}_indices_labels_umap.csv",index_col=0)

In [37]:
adata.obs = adata.obs.join(indices,how='left')

In [38]:
adata.strings_to_categoricals()

In [39]:
groupby='Classification_L2'

In [40]:
adata.obs[groupby].value_counts()

Classification_L2
Granulocytes                 28436662
CD4+ T cells                  6462729
Non-MAIT/NKT CD8+ T cells     5037308
ILCs                          4934984
Monocytes                     3575647
B cells                       1817869
γδ T cells                     933220
DC                             305061
MAIT                           280148
NKT                             80964
HSPC                            18549
DN T cells                       8270
Platelets                        7428
Name: count, dtype: int64

In [44]:
adata.obsm['X_umap'] = indices[['UMAP_1', 'UMAP_2']].values

In [41]:
adata.obs[groupby]=adata.obs[groupby].cat.reorder_categories([
    'CD4+ T cells',
    'Non-MAIT/NKT CD8+ T cells',
    'MAIT',
    'γδ T cells',
    'NKT',
    'DN T cells',
    'ILCs',

    'B cells',
    'Platelets',
    'Granulocytes',
    'Monocytes',
    'DC',
    'HSPC'
])

In [48]:
color_dict={
    'B cells':mcolors.to_rgb('#6495ED'),
    
    'CD4+ T cells':mcolors.to_rgb('#fffac9'),
    'Non-MAIT/NKT CD8+ T cells':mcolors.to_rgb('#CBE5DE'),
    'MAIT':mcolors.to_rgb('#EED0E0'),
    'γδ T cells':mcolors.to_rgb('#f0c1f5'), 
    'NKT':mcolors.to_rgb('#AED0DF'),
    'DN T cells':mcolors.to_rgb('#D2EBC8'), 

    'Monocytes':mcolors.to_rgb('#7DBFA7'),
    'DC':mcolors.to_rgb('#EE934E'), 

    'ILCs':mcolors.to_rgb('#ffe2db'), 
    
    'Granulocytes':mcolors.to_rgb('#c496ff'), 
    
    'HSPC':mcolors.to_rgb('#D1352B'),
    
    'Platelets':mcolors.to_rgb('#B383B9'),
}

In [49]:
print(f"Plot umap,{datetime.now()}")
sc.pl.umap(
    adata,
    color=groupby,
    legend_fontsize=8,ncols=3,frameon=False,
    size=0.05,#legend_loc='on data',
    save=f'_{dataset}_{groupby}_legend',show=False,
    palette=color_dict
)

Plot umap,2025-12-21 16:49:13.378812


<Axes: title={'center': 'Classification_L2'}, xlabel='UMAP1', ylabel='UMAP2'>

In [50]:
print(f"Plot umap,{datetime.now()}")
sc.pl.umap(
    adata,
    color=groupby,
    legend_fontsize=8,ncols=3,frameon=False,
    size=0.05,legend_loc='on data',
    save=f'_{dataset}_{groupby}',show=False,
    palette=color_dict
)

Plot umap,2025-12-21 16:56:23.076837


<Axes: title={'center': 'Classification_L2'}, xlabel='UMAP1', ylabel='UMAP2'>

# 3. Process adt

In [28]:
adt.obsm = adata.obsm

In [29]:
adt.X.max()

np.float32(3863.0)

In [30]:
adt.layers["clr"] = mu.prot.pp.clr(adt,inplace= False).X.copy()

In [31]:
adt.write(f"{obj_path}{dataset}_scADT_count.h5ad",compression="gzip")

In [32]:
import session_info
session_info.show()

/home/liyanguo/anaconda3/envs/R45/lib/python3.13/site-packages/session_info/main.py:213: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  mod_version = _find_version(mod.__version__)
